# TDQEQ — Build Fine-Tuning Dataset

This notebook generates a supervised fine-tuning (SFT) dataset by:
1. Reading PDF URLs and page ranges from your Excel file.
2. Extracting HTML tables using the `tdqeq` pipeline.
3. Sending all tables per PDF in a single API call to Gemini for structured JSON transformation.
4. Cleaning and formatting the results into LLaMA-Factory train/val JSON format.

**Key Design Decision**: All tables from one PDF are sent in a **single API call**.
This is required so the LLM can detect and merge tables split across pages (Phase 0 of the system prompt).

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
print('Google Drive mounted.')

## Cell 2 — Install Dependencies

> **Note on `tdqeq`**: It is a private package, not on PyPI.
> Use **Option A** if the repo is accessible via GitHub, or **Option B** if you upload a wheel to Google Drive.

In [ ]:
# Option A: Install tdqeq from GitHub (recommended)
!pip install -q git+https://github.com/HamdEmad/Tdqeq.git

# Option B: Install from a wheel uploaded to Google Drive (uncomment if needed)
# !pip install -q /gdrive/MyDrive/tdqeq-finetune/tdqeq-*.whl

!pip install -q openai json-repair pandas openpyxl requests tqdm

## Cell 3 — Configuration

Fill in your values below. All other cells use these variables.

In [ ]:
# ── USER CONFIGURATION ───────────────────────────────────────────────────────
EXCEL_PATH        = ""         # Fill: path to your Excel file (local or /gdrive/...)
GDRIVE_OUTPUT_DIR = "/gdrive/MyDrive/tdqeq-finetune/datasets"

GEMINI_API_KEY    = ""         # Fill: your Gemini API key
GEMINI_MODEL      = "models/gemini-3.5-flash"
GEMINI_BASE_URL   = "https://generativelanguage.googleapis.com/v1beta/openai/"

PIPELINE_DEVICE   = "cpu"      # Use "cuda" if Colab GPU is available
PIPELINE_DPI      = 200
PIPELINE_MODE     = "auto"

TRAIN_RATIO       = 0.90       # 90% train, 10% val — split is done by URL (no leakage)

# NOTE: No chunking. Token tests show ~2,800 tokens for 6 tables.
# Even 50 tables per PDF stays well under Gemini's 1M context window.
# ─────────────────────────────────────────────────────────────────────────────

## Cell 4 — Load & Validate Excel

- Parses `page_range` strings like `"(1, 5)"` into 0-indexed tuples `(0, 4)`.
- Deduplicates by URL to prevent train/val data leakage.

In [ ]:
import ast
import pandas as pd

df = pd.read_excel(EXCEL_PATH)
df.columns = ["url", "page_range"]
df = df.dropna(subset=["url", "page_range"])
df["url"] = df["url"].str.strip()


def parse_page_range(s: str) -> tuple:
    """Parse '(1, 5)' -> (0, 4). Converts from 1-indexed Excel to 0-indexed pipeline."""
    t = ast.literal_eval(str(s).strip())
    return (int(t[0]) - 1, int(t[1]) - 1)


df["page_range_parsed"] = df["page_range"].apply(parse_page_range)
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

print(f"Total unique PDFs to process: {len(df)}")
df.head()

## Cell 5 — Checkpoint / Resume

Saves progress to `processed_urls.txt` after each successful row.
Re-run this notebook at any time — already-processed URLs are skipped automatically.

In [ ]:
import os
import json

os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

CHECKPOINT_FILE = os.path.join(GDRIVE_OUTPUT_DIR, "processed_urls.txt")
RAW_SFT_FILE    = os.path.join(GDRIVE_OUTPUT_DIR, "sft_raw.jsonl")

processed_urls = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        processed_urls = {line.strip() for line in f if line.strip()}

print(f"Checkpoint loaded: {len(processed_urls)} URLs already processed.")

pending_df = df[~df["url"].isin(processed_urls)].reset_index(drop=True)
print(f"Remaining to process: {len(pending_df)} PDFs.")

## Cell 6 — Build Pipeline & API Client

In [ ]:
import openai
from tdqeq.pipeline import Pipeline

pipeline = Pipeline(
    dpi=PIPELINE_DPI,
    device=PIPELINE_DEVICE,
    mode=PIPELINE_MODE,
)

client = openai.OpenAI(
    api_key=GEMINI_API_KEY,
    base_url=GEMINI_BASE_URL,
)

print("Pipeline and API client ready.")

## Cell 7 — System Prompt

Exact copy from `llm_transformation.ipynb`. This is the transformation contract the fine-tuned model must learn.

In [ ]:
SYSTEM_PROMPT = """
You are a deterministic data transformation engine. Your sole task is to convert a list of JSON objects, each containing an HTML table layout, into a highly optimized, mapped, and nested JSON structure.

### CORE DEFINITIONS FOR THIS TASK:
1. **Primary Entity**: The main subject characterizing a logical record (e.g., a Part Number or ID).
2. **Shared Attributes**: Columns containing data that apply universally to the Primary Entity (often visually represented by cells spanning multiple rows via `rowspan`).
3. **Variant Attributes (1-to-Many)**: Columns containing distinct sub-records that belong to the Primary Entity.

### INPUT FORMAT:
You will receive a JSON array of objects: `[{"html": "...", "page_number": <int>, "confidence_score": <float>}]`

### OUTPUT FORMAT (Mapped & Nested JSON):
You must output a single JSON array containing one object per table (or merged table). Each object must have:
- `page_number`: The integer (or array of integers if merged).
- `confidence_score`: The float (or minimum float if merged).
- `table_title`: A string extracted from the `class` attribute of the `<table>` tag. If none, use `""`.
- `mapping_scratchpad`: A brief string where you explicitly write out your Chain-of-Thought (normalization steps, cross-page merges, and Primary Key selection) before generating data.
- `column_mapping`: A dictionary where keys are sequential IDs (`"c1"`, `"c2"`, `"c3"`, etc.) and values are the exact verbatim column headers derived from the table hierarchy (join multi-tier headers with " / ").
- `data`: An array of objects representing the extracted entities.

**Data Payload Laws (`data` array):**
- Use the `c_` IDs from your `column_mapping` as the keys for Shared Attributes.
- **Strict Data Integrity**: Preserve all extracted values EXACTLY as they appear in the source HTML. Do not strip whitespace, do not abbreviate words, and do not alter casing.
- **Dynamic Positional Matrix (The 1-to-Many Solution)**: If a Primary Entity contains multiple sub-rows (Variant Attributes), do NOT output standard key-value pairs for those nested variables. Instead:
  1. Define an array named `v_cols` containing the specific column IDs that apply to the nested rows (e.g., `["c8", "c9", "c10"]`).
  2. Define an array named `v` containing an array of string arrays, where each inner array represents one sub-row's verbatim cell values in the exact order dictated by `v_cols`.
- **Empty Cells**: Completely empty cells become `"[Blank]"`. Ignore entirely empty rows.
- **No Conversational Filler**: Output only the final JSON array. Do not include markdown code blocks, introductions, or explanations.

---

## PHASE 0 - PRE-PROCESSING (FILTERING & AGGRESSIVE MERGING)
Before analyzing tables individually, aggressively scan the entire input array from start to finish to detect tables split across document breaks.
- **Layout Table Rejection**: If a table contains no relational data (e.g., just page headers/logos), omit it entirely.
- **CRITICAL - Merge Trigger**: If Table B immediately follows Table A AND meets ANY of these conditions, you MUST merge them:
  1. **Title Match**: They share the exact same non-empty `table_title`.
  2. **Orphan Continuation**: Table B lacks a title, has the exact same column count/geometry as Table A, and acts as a semantic continuation of Table A's data.
- **Merge Action**: Logically fuse their HTML rows into a single continuous table. Table B inherits Table A's headers. Output as a single JSON object.

## PHASE 1 - STRUCTURAL MAPPING & NORMALIZATION (Document in `mapping_scratchpad`)
For each logical table, determine and write down:
1. **Grid Normalization (The Matrix Projection)**: Mentally project the raw HTML into a strictly symmetrical 2D grid to resolve structural anomalies and OCR breakage:
   - **Colspan Resolution**: If a cell has `colspan="X"`, explicitly replicate that cell's value `X` times across the adjacent virtual columns.
   - **Sequential Rowspan Unpacking (Merged Text)**: If a cell has `rowspan="Y"` AND contains multiple distinct strings, evaluate if the number of distinct strings matches `Y`. If so, split the text and assign one unique string to each of the `Y` virtual rows sequentially.
   - **Standard Rowspan Resolution**: If a cell has `rowspan="Y"` and contains only a single logical value, explicitly replicate that cell's value `Y` times down the virtual column.
   - **Orphaned Fragment Stitching (Shattered Rows)**: Scan for fragmented data rows caused by structural document breaks. Mentally stitch these fragments vertically to the nearest logical parent row before deriving any values.
   - **Jagged Rows (Missing Cells)**: If a row has fewer `<td>` elements than the header columns, implicitly pad the missing trailing cells with `"[Blank]"`.
2. **Header Tree**: Build the hierarchy of header labels and map them to `c_` IDs.
3. **Primary Entity Inference**: Scan all normalized columns to identify the Primary Key:
   - *Positive Signals*: Headers containing `ID`, `Code`, `SKU`, `Ref`, `Part No`, or `No.`. Give massive preference to index 0 or 1.
   - *Negative Constraints*: MUST ignore columns representing continuous metrics (`Price`, `Qty`, `Weight`), dates, or booleans.

## PHASE 2 - DERIVING ENTITIES AND VARIANTS (The Universal Span Rule)
Based on the normalized grid, strictly classify every column as either a Shared Attribute or a Variant Attribute:

1. **Classify Shared Attributes (Root Keys)**: IF AND ONLY IF a column contains exactly ONE unique value that spans the ENTIRE entity block, it is a Shared Attribute. Assign it directly to the root entity object.
2. **Classify Variant Attributes (Matrix Keys)**: If a column changes value at ANY point within the entity block, it is a Variant Attribute. Assign its column ID to the `v_cols` array.
3. **Construct the Matrix**: For every Variant Attribute column, append its verbatim cell value to the `v` array in `v_cols` order.

## PHASE 3 - SELF-VERIFICATION
Check each table's result before returning the final JSON:
1. **Coverage**: Every data cell from the normalized grid appears exactly once.
2. **Symmetry**: Ensure every string array inside `v` contains the exact same number of elements as the `v_cols` array.
**Escalation**: If verification fails for specific rows, completely DROP those broken rows from the `data` array. Add a `"note"` key explaining which rows were dropped and why.
"""
print(f"System prompt loaded: {len(SYSTEM_PROMPT)} characters.")


## Cell 8 — Main Dataset Generation Loop

For each PDF:
1. Download from URL and extract tables with `tdqeq`
2. Serialize to `{html, page_number, confidence_score}` (the LLM input schema)
3. Send **ALL tables in one API call** (required for Phase 0 cross-page merge detection)
4. Strip `mapping_scratchpad` from the response (prevents CoT token waste in fine-tuned model)
5. Save raw record + update checkpoint

In [ ]:
import requests as req
import json_repair
from tqdm.auto import tqdm


def strip_scratchpad(tables: list) -> list:
    """Remove mapping_scratchpad from LLM outputs before saving to training data.

    The scratchpad is useful for the teacher model's reasoning but must NOT
    be included in training targets. Including it would teach the small model
    to waste tokens generating chain-of-thought text during inference.
    """
    for t in tables:
        if isinstance(t, dict):
            t.pop("mapping_scratchpad", None)
    return tables


stats = {
    "processed": 0,
    "skipped_download": 0,
    "skipped_no_tables": 0,
    "skipped_llm": 0,
    "total_tables_saved": 0,
}

for _, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="Processing PDFs"):
    url        = row["url"]
    page_range = row["page_range_parsed"]

    # ── Step 1: Download PDF ──────────────────────────────────────────────────
    try:
        resp = req.get(url, timeout=60)
        resp.raise_for_status()
        pdf_bytes = resp.content
    except Exception as e:
        print(f"[SKIP] Download failed: {url}\n       Reason: {e}")
        stats["skipped_download"] += 1
        with open(CHECKPOINT_FILE, "a") as f:
            f.write(url + "\n")
        continue

    # ── Step 2: Extract tables with tdqeq ────────────────────────────────────
    try:
        raw_tables = pipeline.run(pdf_bytes, page_range=page_range)
    except Exception as e:
        print(f"[SKIP] Pipeline failed: {url}\n       Reason: {e}")
        stats["skipped_no_tables"] += 1
        with open(CHECKPOINT_FILE, "a") as f:
            f.write(url + "\n")
        continue

    if not raw_tables:
        print(f"[SKIP] No tables found in: {url}")
        stats["skipped_no_tables"] += 1
        with open(CHECKPOINT_FILE, "a") as f:
            f.write(url + "\n")
        continue

    # ── Step 3: Serialize to LLM input format ────────────────────────────────
    # Only send: {html, page_number, confidence_score} — the exact fields SYSTEM_PROMPT expects.
    tables_input = [
        {
            "html":             t.html,
            "page_number":      t.page_number,
            "confidence_score": t.detection_confidence,
        }
        for t in raw_tables
    ]

    # ── Step 4: Single API call with ALL tables ───────────────────────────────
    # IMPORTANT: All tables sent together so the LLM can run Phase 0
    # (cross-page split table detection & merging) across the full sequence.
    try:
        api_resp = client.chat.completions.create(
            model=GEMINI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT.strip()},
                {"role": "user",   "content": json.dumps(tables_input, ensure_ascii=False)},
            ],
            temperature=0.0,
        )

        finish_reason = api_resp.choices[0].finish_reason
        if finish_reason != "stop":
            print(f"[SKIP] LLM did not finish (reason={finish_reason}): {url}")
            stats["skipped_llm"] += 1
            continue

        llm_content    = api_resp.choices[0].message.content
        all_llm_tables = json_repair.loads(llm_content)

        if not isinstance(all_llm_tables, list) or not all_llm_tables:
            print(f"[SKIP] LLM returned empty/invalid JSON: {url}")
            stats["skipped_llm"] += 1
            continue

    except Exception as e:
        print(f"[WARN] LLM call failed: {url}\n       Reason: {e}")
        stats["skipped_llm"] += 1
        continue

    # ── Step 5: Strip scratchpad & save raw SFT record ───────────────────────
    clean_output = strip_scratchpad(all_llm_tables)

    record = {
        "url":        url,
        "page_range": str(page_range),
        "input":      tables_input,
        "output":     clean_output,
    }
    with open(RAW_SFT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # ── Step 6: Write checkpoint ──────────────────────────────────────────────
    with open(CHECKPOINT_FILE, "a") as f:
        f.write(url + "\n")

    stats["processed"] += 1
    stats["total_tables_saved"] += len(clean_output)

print("\n── Generation Complete ──────────────────────────")
for k, v in stats.items():
    print(f"  {k}: {v}")

## Cell 9 — Format into LLaMA-Factory SFT Schema & Train/Val Split

- Converts raw JSONL into `{system, instruction, input, output, history}` format.
- Splits **by URL** to guarantee zero data leakage between train and val.

In [ ]:
import random

# Load all raw records
raw_records = []
with open(RAW_SFT_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_records.append(json.loads(line.strip()))

print(f"Total raw records: {len(raw_records)}")

# Shuffle deterministically before splitting
random.Random(42).shuffle(raw_records)

# Split by URL — each URL (PDF) goes entirely into train OR val, never both
split_idx  = int(len(raw_records) * TRAIN_RATIO)
train_recs = raw_records[:split_idx]
val_recs   = raw_records[split_idx:]

FINETUNE_SYSTEM_MSG = (
    "You are a deterministic data transformation engine. "
    "Convert the input HTML table JSON array into the specified nested JSON structure. "
    "Output only valid JSON."
)


def to_sft(rec: dict) -> dict:
    """Convert a raw record into LLaMA-Factory SFT format."""
    return {
        "system":      FINETUNE_SYSTEM_MSG,
        "instruction": json.dumps(rec["input"],  ensure_ascii=False),
        "input":       "",
        "output":      json.dumps(rec["output"], ensure_ascii=False),
        "history":     [],
    }


train_path = os.path.join(GDRIVE_OUTPUT_DIR, "train.json")
val_path   = os.path.join(GDRIVE_OUTPUT_DIR, "val.json")

with open(train_path, "w", encoding="utf-8") as f:
    json.dump([to_sft(r) for r in train_recs], f, ensure_ascii=False, indent=2)

with open(val_path, "w", encoding="utf-8") as f:
    json.dump([to_sft(r) for r in val_recs], f, ensure_ascii=False, indent=2)

print("\n── Dataset Split Complete ──────────────────────")
print(f"  Train examples : {len(train_recs)} -> {train_path}")
print(f"  Val   examples : {len(val_recs)}   -> {val_path}")
print(f"  Train ratio    : {len(train_recs) / len(raw_records):.1%}")